In [1]:
import os
os.chdir("..")

In [2]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from structures.network import VectorizedNetwork, OOPNetwork
from structures.base import accuracy, f1_score_macro

In [3]:
# Data setup
n_classes = 3
n_features = 5
last_layer_size = 1 if n_classes == 2 else n_classes

In [4]:
X, y = make_classification(n_samples=500, n_features=n_features, n_informative=5, n_classes=n_classes,
                           n_redundant=0, n_clusters_per_class=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y.reshape(-1, 1), test_size=0.2, random_state=42)

In [5]:
# Network parameters
layer_sizes = [n_features, 10, 20, last_layer_size]
epochs = 200
lr = 0.05

### Optimized Layer

In [6]:
# Create network
network = OOPNetwork(layer_sizes, task="classification", activation="sigmoid")

In [9]:
# Training network
for epoch in range(epochs):
    loss = 0
    for xi, yi in zip(X_train, y_train):
        network.backward(xi, yi, lr)
        pred_vec = network.forward(xi)  # full softmax vector
        # One-hot encode target
        one_hot = np.zeros_like(pred_vec)
        one_hot[int(yi)] = 1
        # Cross-entropy loss
        loss += -np.sum(one_hot * np.log(pred_vec + 1e-9))  # add epsilon for numerical stability

    if epoch % 20 == 0 or epoch == epochs - 1:
        predicted = np.array([network.predict(x) for x in X_test])
        acc = accuracy(y_test, predicted)
        f1 = f1_score_macro(y_test, predicted)
        print(f"loss: {loss}, accuracy: {acc}, f1: {f1}.")

/tmp/ipykernel_7623/3403249022.py:9: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  one_hot[int(yi)] = 1


loss: 386.1886109356675, accuracy: 0.32, f1: 0.16161616161616163.
loss: 266.1277446922743, accuracy: 0.72, f1: 0.7046376811594204.
loss: 249.66429045862526, accuracy: 0.75, f1: 0.7290914455106141.
loss: 197.94831295225848, accuracy: 0.83, f1: 0.823666714302241.
loss: 144.75704301800232, accuracy: 0.8, f1: 0.7965419636462845.
loss: 94.89844273046727, accuracy: 0.85, f1: 0.8482566000628252.
loss: 56.14608526396935, accuracy: 0.85, f1: 0.8487352918539847.
loss: 39.02775242291693, accuracy: 0.86, f1: 0.8590132827324478.
loss: 29.573622977610395, accuracy: 0.86, f1: 0.8601505224333704.
loss: 27.126374582502613, accuracy: 0.86, f1: 0.8601505224333704.
loss: 25.96502557886052, accuracy: 0.86, f1: 0.8601505224333704.


### Vectorized approach

In [29]:
# Instantiate network
vector_net = VectorizedNetwork(layer_sizes, task="classification", activation="sigmoid")

In [28]:
# Training vectorized network
for epoch in range(epochs):
    loss = 0
    for xi, yi in zip(X_train, y_train):
        vector_net.backward(xi, yi, lr)
        pred_vec = vector_net.forward(xi)[0][-1]  # full softmax vector
        # One-hot encode target
        one_hot = np.zeros_like(pred_vec)
        one_hot[int(yi)] = 1
        # Cross-entropy loss
        loss += -np.sum(one_hot * np.log(pred_vec + 1e-9))  # add epsilon for numerical stability

    if epoch % 20 == 0 or epoch == epochs - 1:
        predicted = np.array([vector_net.predict(x) for x in X_test])
        acc = accuracy(y_test, predicted)
        f1 = f1_score_macro(y_test, predicted)
        print(f"loss: {loss}, accuracy: {acc}, f1: {f1}.")

/tmp/ipykernel_16881/3502804789.py:9: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  one_hot[int(yi)] = 1


loss: 386.0906189304096, accuracy: 0.32, f1: 0.16161616161616163.
loss: 266.4218399732474, accuracy: 0.72, f1: 0.7046376811594204.
loss: 255.78559513654307, accuracy: 0.74, f1: 0.7206526269452133.
loss: 205.0624021961917, accuracy: 0.84, f1: 0.8337690631808279.
loss: 168.17765769910474, accuracy: 0.83, f1: 0.8252542737218431.
loss: 123.93056586179844, accuracy: 0.82, f1: 0.8194090539441583.
loss: 75.51283509196844, accuracy: 0.89, f1: 0.8899442183024272.
loss: 50.46257676836061, accuracy: 0.84, f1: 0.836687306501548.
loss: 38.63645088659336, accuracy: 0.86, f1: 0.8562929445282386.
loss: 33.596441583748174, accuracy: 0.86, f1: 0.8562929445282386.
loss: 30.731818031128128, accuracy: 0.85, f1: 0.8464731347084288.
